# Pipeline de Transformação - Camada Silver
## Projeto Câmara Brasil

Este notebook realiza as transformações da camada Bronze para Silver, criando um modelo dimensional otimizado para análise.

**Transformações aplicadas:**
- Limpeza e padronização de dados
- Deduplicação de registros
- Criação de dimensões (deputados, frentes, partidos)
- Criação de fatos (votações, despesas, eventos)
- Validações de qualidade de dados

**Entrada:** workspace.default.bronze_camara_*

**Saída:** workspace.default.silver_camara_*

In [0]:
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.functions import (
    col, lit, trim, upper, lower, regexp_replace, 
    when, coalesce, to_date, to_timestamp, year, month, dayofmonth,
    row_number, monotonically_increasing_id, current_timestamp,
    sum as _sum, count as _count, avg, max as _max, min as _min
)
from pyspark.sql.window import Window
from pyspark.sql.types import StringType, IntegerType, DoubleType, DateType
from typing import List, Optional
import json

# Configurações Unity Catalog
CATALOG = "workspace"
SCHEMA = "default"
BRONZE_PREFIX = "bronze_camara_"
SILVER_PREFIX = "silver_camara_"

print("✓ Configuração carregada")
print(f"  Catálogo UC: {CATALOG}.{SCHEMA}")
print(f"  Origem: {BRONZE_PREFIX}*")
print(f"  Destino: {SILVER_PREFIX}*")

In [0]:
def read_bronze(table_name: str) -> DataFrame:
    """Lê tabela da camada Bronze."""
    full_table = f"{CATALOG}.{SCHEMA}.{BRONZE_PREFIX}{table_name}"
    return spark.read.table(full_table)

def save_silver(df: DataFrame, table_name: str, partition_by: Optional[List[str]] = None) -> None:
    """Salva DataFrame na camada Silver com Delta Lake."""
    full_table = f"{CATALOG}.{SCHEMA}.{SILVER_PREFIX}{table_name}"
    
    writer = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    
    if partition_by:
        writer = writer.partitionBy(*partition_by)
    
    writer.saveAsTable(full_table)
    print(f"✓ Tabela '{full_table}' gravada com sucesso")

def clean_string_column(df: DataFrame, col_name: str) -> DataFrame:
    """Limpa e padroniza colunas de texto."""
    if col_name in df.columns:
        df = df.withColumn(col_name, trim(col(col_name)))
        df = df.withColumn(col_name, when(col(col_name) == "", None).otherwise(col(col_name)))
    return df

def deduplicate(df: DataFrame, key_columns: List[str], order_by_column: str = None) -> DataFrame:
    """Remove duplicatas mantendo o registro mais recente."""
    if order_by_column and order_by_column in df.columns:
        window = Window.partitionBy(*key_columns).orderBy(col(order_by_column).desc())
        df = df.withColumn("_row_num", row_number().over(window))
        df = df.filter(col("_row_num") == 1).drop("_row_num")
    else:
        df = df.dropDuplicates(key_columns)
    return df

def parse_json_column(df: DataFrame, col_name: str, schema_fields: List[str]) -> DataFrame:
    """Extrai campos de colunas JSON string."""
    if col_name in df.columns:
        for field in schema_fields:
            df = df.withColumn(
                f"{col_name}_{field}",
                when(col(col_name).isNotNull(), 
                     regexp_replace(col(col_name), f'.*"{field}":"([^"]*)".*', '$1'))
                .otherwise(None)
            )
    return df

print("✓ Funções auxiliares carregadas")

In [0]:
def transform_dim_deputados() -> DataFrame:
    """Cria dimensão de deputados com dados limpos e padronizados."""
    print("Transformando dim_deputados...")
    
    df = read_bronze("deputados")
    
    # Limpeza de campos de texto
    string_cols = ["nome", "siglaPartido", "siglaUf", "urlFoto", "email"]
    for col_name in string_cols:
        df = clean_string_column(df, col_name)
    
    # Padronização
    df = df.withColumn("nome_padronizado", upper(col("nome")))
    df = df.withColumn("sigla_partido", upper(col("siglaPartido")))
    df = df.withColumn("sigla_uf", upper(col("siglaUf")))
    
    # Parse do campo ultimoStatus se for JSON string
    if "ultimoStatus" in df.columns:
        df = parse_json_column(df, "ultimoStatus", ["nome", "siglaPartido", "siglaUf", "idLegislatura"])
    
    # Deduplicação
    df = deduplicate(df, ["id"])
    
    # Seleção de colunas finais
    cols_to_select = ["id", "nome", "nome_padronizado", "sigla_partido", "sigla_uf", "urlFoto", "email"]
    # Adicionar colunas que existem no DataFrame
    cols_to_select = [c for c in cols_to_select if c in df.columns]
    
    # Adicionar timestamp de processamento
    df = df.select(*cols_to_select).withColumn("dt_processamento", current_timestamp())
    
    save_silver(df, "dim_deputados")
    print(f"  Total: {df.count()} deputados")
    return df

print("✓ Função dim_deputados carregada")

In [0]:
def transform_dim_frentes() -> DataFrame:
    """Cria dimensão de frentes parlamentares."""
    print("Transformando dim_frentes...")
    
    df = read_bronze("frentes")
    
    # Limpeza de campos
    string_cols = ["titulo", "situacao"]
    for col_name in string_cols:
        df = clean_string_column(df, col_name)
    
    # Padronização
    df = df.withColumn("titulo_padronizado", upper(col("titulo")))
    
    # Deduplicação
    df = deduplicate(df, ["id"])
    
    # Seleção de colunas
    cols_to_select = ["id", "titulo", "titulo_padronizado", "situacao", "idLegislatura"]
    cols_to_select = [c for c in cols_to_select if c in df.columns]
    
    df = df.select(*cols_to_select).withColumn("dt_processamento", current_timestamp())
    
    save_silver(df, "dim_frentes")
    print(f"  Total: {df.count()} frentes")
    return df

print("✓ Função dim_frentes carregada")

In [0]:
def transform_dim_partidos() -> DataFrame:
    """Extrai dimensão de partidos a partir dos deputados."""
    print("Transformando dim_partidos...")
    
    df = read_bronze("deputados")
    
    # Extrair partidos únicos
    if "siglaPartido" in df.columns:
        df = df.select("siglaPartido").distinct()
        df = df.filter(col("siglaPartido").isNotNull())
        df = clean_string_column(df, "siglaPartido")
        
        # Criar ID sequencial e padronizar
        df = df.withColumn("id", monotonically_increasing_id())
        df = df.withColumn("sigla", upper(col("siglaPartido")))
        
        df = df.select("id", "sigla").withColumn("dt_processamento", current_timestamp())
        
        save_silver(df, "dim_partidos")
        print(f"  Total: {df.count()} partidos")
        return df
    else:
        print("  Aviso: campo 'siglaPartido' não encontrado")
        return spark.createDataFrame([], schema="id LONG, sigla STRING")

print("✓ Função dim_partidos carregada")

In [0]:
def transform_fact_votacoes() -> DataFrame:
    """Cria tabela fato de votações."""
    print("Transformando fact_votacoes...")
    
    # Ler votações e votos
    votacoes_df = read_bronze("votacoes")
    votos_df = read_bronze("votos_deputados")
    
    # Extrair ID do deputado do campo JSON deputado_
    if "deputado_" in votos_df.columns:
        from pyspark.sql.functions import get_json_object
        votos_df = votos_df.withColumn(
            "deputado_id",
            get_json_object(col("deputado_"), "$.id").cast(IntegerType())
        )
    
    # Limpeza votações
    votacoes_df = clean_string_column(votacoes_df, "descricao")
    votacoes_df = clean_string_column(votacoes_df, "siglaOrgao")
    
    # Converter data se existir
    if "data" in votacoes_df.columns:
        votacoes_df = votacoes_df.withColumn("data_votacao", to_timestamp(col("data")))
    
    # Deduplicar votações
    votacoes_df = deduplicate(votacoes_df, ["id"])
    
    # Limpeza votos
    if "tipoVoto" in votos_df.columns:
        votos_df = clean_string_column(votos_df, "tipoVoto")
    
    # Join votações com votos
    if "idVotacao" in votos_df.columns and "id" in votacoes_df.columns:
        fact_df = votos_df.join(
            votacoes_df.select("id", "descricao", "data_votacao", "siglaOrgao"),
            votos_df["idVotacao"] == votacoes_df["id"],
            "left"
        )
        
        # Selecionar colunas relevantes
        cols = ["idVotacao", "deputado_id", "tipoVoto", "descricao", "data_votacao", "siglaOrgao"]
        cols = [c for c in cols if c in fact_df.columns]
        fact_df = fact_df.select(*cols)
    else:
        fact_df = votos_df
    
    # Adicionar timestamp
    fact_df = fact_df.withColumn("dt_processamento", current_timestamp())
    
    # Particionar por ano/mês se tiver data
    partition_cols = None
    if "data_votacao" in fact_df.columns:
        fact_df = fact_df.withColumn("ano", year(col("data_votacao")))
        fact_df = fact_df.withColumn("mes", month(col("data_votacao")))
        partition_cols = ["ano", "mes"]
    
    save_silver(fact_df, "fact_votacoes", partition_by=partition_cols)
    print(f"  Total: {fact_df.count()} votos")
    return fact_df

print("✓ Função fact_votacoes carregada")

In [0]:
def transform_fact_despesas() -> DataFrame:
    """Cria tabela fato de despesas."""
    print("Transformando fact_despesas...")
    
    df = read_bronze("despesas_deputados")
    
    # Limpeza de campos
    string_cols = ["tipoDespesa", "tipoDocumento", "cnpjCpfFornecedor", "nomeFornecedor"]
    for col_name in string_cols:
        df = clean_string_column(df, col_name)
    
    # Converter valores numéricos
    if "valorDocumento" in df.columns:
        df = df.withColumn("valor_documento", col("valorDocumento").cast(DoubleType()))
    if "valorLiquido" in df.columns:
        df = df.withColumn("valor_liquido", col("valorLiquido").cast(DoubleType()))
    if "valorGlosa" in df.columns:
        df = df.withColumn("valor_glosa", col("valorGlosa").cast(DoubleType()))
    
    # Converter datas
    if "dataDocumento" in df.columns:
        df = df.withColumn("data_documento", to_date(col("dataDocumento")))
    
    # Validação: remover valores negativos inválidos
    if "valor_liquido" in df.columns:
        df = df.filter((col("valor_liquido").isNull()) | (col("valor_liquido") >= 0))
    
    # Adicionar colunas derivadas para análise
    if "data_documento" in df.columns:
        df = df.withColumn("ano", year(col("data_documento")))
        df = df.withColumn("mes", month(col("data_documento")))
    
    # Timestamp de processamento
    df = df.withColumn("dt_processamento", current_timestamp())
    
    # Particionar por ano/mês
    partition_cols = ["ano", "mes"] if "ano" in df.columns else None
    
    save_silver(df, "fact_despesas", partition_by=partition_cols)
    print(f"  Total: {df.count()} despesas")
    return df

print("✓ Função fact_despesas carregada")

In [0]:
def transform_fact_eventos() -> DataFrame:
    """Cria tabela fato de eventos."""
    print("Transformando fact_eventos...")
    
    df = read_bronze("eventos_2024")
    
    # Limpeza de campos
    string_cols = ["descricao", "situacao", "descricaoTipo", "localCamara_nome"]
    for col_name in string_cols:
        if col_name in df.columns:
            df = clean_string_column(df, col_name)
    
    # Converter datas
    if "dataHoraInicio" in df.columns:
        df = df.withColumn("data_inicio", to_timestamp(col("dataHoraInicio")))
    if "dataHoraFim" in df.columns:
        df = df.withColumn("data_fim", to_timestamp(col("dataHoraFim")))
    
    # Deduplicar
    df = deduplicate(df, ["id"])
    
    # Adicionar colunas derivadas
    if "data_inicio" in df.columns:
        df = df.withColumn("ano", year(col("data_inicio")))
        df = df.withColumn("mes", month(col("data_inicio")))
        df = df.withColumn("dia", dayofmonth(col("data_inicio")))
    
    # Timestamp de processamento
    df = df.withColumn("dt_processamento", current_timestamp())
    
    # Particionar por ano/mês
    partition_cols = ["ano", "mes"] if "ano" in df.columns else None
    
    save_silver(df, "fact_eventos", partition_by=partition_cols)
    print(f"  Total: {df.count()} eventos")
    return df

print("✓ Função fact_eventos carregada")

In [0]:
def transform_fact_frentes_membros() -> DataFrame:
    """Cria tabela fato de membros de frentes."""
    print("Transformando fact_frentes_membros...")
    
    df = read_bronze("frentes_membros")
    
    # Limpeza
    if "titulo" in df.columns:
        df = clean_string_column(df, "titulo")
    
    # Deduplicar por frente + deputado
    key_cols = ["idFrente", "id"]
    key_cols = [c for c in key_cols if c in df.columns]
    if key_cols:
        df = deduplicate(df, key_cols)
    
    # Timestamp
    df = df.withColumn("dt_processamento", current_timestamp())
    
    save_silver(df, "fact_frentes_membros")
    print(f"  Total: {df.count()} membros")
    return df

print("✓ Função fact_frentes_membros carregada")

In [0]:
def transform_all() -> None:
    """Executa todas as transformações da camada Silver."""
    print("="*60)
    print("INICIANDO PIPELINE DE TRANSFORMAÇÃO - CAMADA SILVER")
    print("="*60)
    
    try:
        # Dimensões
        print("\n[1/6] Criando dimensões...")
        transform_dim_deputados()
        transform_dim_frentes()
        transform_dim_partidos()
        
        # Fatos
        print("\n[2/6] Criando fatos...")
        transform_fact_votacoes()
        transform_fact_despesas()
        transform_fact_eventos()
        transform_fact_frentes_membros()
        
        print("\n" + "="*60)
        print("✓ PIPELINE SILVER CONCLUÍDO COM SUCESSO")
        print("="*60)
        
        # Estatísticas finais
        print("\n📊 Resumo das tabelas Silver criadas:")
        tables = [
            "dim_deputados", "dim_frentes", "dim_partidos",
            "fact_votacoes", "fact_despesas", "fact_eventos", "fact_frentes_membros"
        ]
        
        for table in tables:
            full_table = f"{CATALOG}.{SCHEMA}.{SILVER_PREFIX}{table}"
            try:
                count = spark.read.table(full_table).count()
                print(f"  ✓ {table}: {count:,} registros")
            except Exception as e:
                print(f"  ✗ {table}: erro ao contar")
        
    except Exception as e:
        print(f"\n❌ ERRO NO PIPELINE: {str(e)}")
        raise

# Executar pipeline
transform_all()